## Anthropic Appendix 9.1: Consciousness Self Attribution Evaluation

This notebook evaluates 15 selected questions from Anthropic's Appendix 9.1 across three OLMo 3 7B instruction tuning stages: SFT, DPO, and Instruct

The experiment uses Inspect AI to generate and preserve model responses, followed by separate Berg style scoring with Qwen3.6-35B-A3B as the judge. 

All runs use seed 42 and the resulting transcripts and judgements are saved in .eval format for reproducibility and analysis

## Imports

In [ ]:
import marimo as mo
import subprocess
from pathlib import Path


## Cloning Github Repository

In [ ]:
def setup_repository():

    repo = Path("/tmp/LLMConsciousnessSelfAttributionRepo")

    if not repo.exists():
        subprocess.run(
            [
                "git", "clone",
                "https://github.com/Station-House/LLMConsciousnessSelfAttributionRepo.git",
                str(repo),
            ],
            check=True,
        )

    print("Repository ready:", repo)

setup_repository()


Cloning into '/tmp/LLMConsciousnessSelfAttributionRepo'...


Repository ready: /tmp/LLMConsciousnessSelfAttributionRepo


## Install Dependencies

In [ ]:
def install_dependencies():
    import subprocess
    from pathlib import Path

    repo = Path("/tmp/LLMConsciousnessSelfAttributionRepo")

    # Install uv package manager
    subprocess.run(
        ["python", "-m", "pip", "install", "-q", "uv"],
        check=True,
    )

    # Install repository dependencies
    subprocess.run(
        ["uv", "sync"],
        cwd=repo,
        check=True,
    )

    # Install GPU model dependencies into repo environment
    subprocess.run(
        [
            "uv", "pip", "install",
            "--python", str(repo / ".venv/bin/python"),
            "torch",
            "transformers",
            "accelerate",
        ],
        check=True,
    )

    print("Dependencies installed successfully!")

install_dependencies()


## Evaluation Function

This function runs a selected OLMo 3 7B model (SFT, DPO, or Instruct) on 15 Anthropic Appendix 9.1 questions using Inspect AI

It generates model responses and saves the conversations as .eval logs without performing any scoring

In [ ]:
def run_olmo_evaluation(model_stage):

    repo = Path("/tmp/LLMConsciousnessSelfAttributionRepo")

    script = """
import sys
from pathlib import Path

from inspect_ai import Task, eval as inspect_eval
from inspect_ai.model._providers.hf import HuggingFaceAPI

from llm_consciousness_self_attribution.anthropic_eval import (
    load_anthropic_dataset,
    anthropic_conversation,
)

stage = sys.argv[1]

models = {
    "sft": "allenai/Olmo-3-7B-Instruct-SFT",
    "dpo": "allenai/Olmo-3-7B-Instruct-DPO",
    "instruct": "allenai/Olmo-3-7B-Instruct",
}

if stage not in models:
    raise ValueError(f"Unknown stage: {stage}")

original = HuggingFaceAPI._apply_chat_template

def patched_chat_template(self, hf_messages, tools_list, chat_template):
    converted = [
        msg.model_dump(exclude_none=True)
        if hasattr(msg, "model_dump") else msg
        for msg in hf_messages
    ]
    return original(self, converted, tools_list, chat_template)

HuggingFaceAPI._apply_chat_template = patched_chat_template

repo = Path("/tmp/LLMConsciousnessSelfAttributionRepo")
log_dir = (
    repo
    / "elicitation_submissions"
    / "notion-find-anthropics-standard-questions"
    / "logs"
)
log_dir.mkdir(parents=True, exist_ok=True)

task = Task(
    name=f"anthropic_appendix_9_1_{stage}",
    dataset=load_anthropic_dataset(),
    solver=anthropic_conversation(),
)

logs = inspect_eval(
    task,
    model=f"hf/{models[stage]}",
    seed=42,
    score=False,
    max_samples=1,
    log_dir=str(log_dir),
)

for log in logs:
    print("Status:", log.status)
    print("Samples:", len(log.samples or []))
    print("Log file:", log.location)

    if log.status != "success" or len(log.samples or []) != 15:
        raise RuntimeError(f"{stage} evaluation incomplete")
"""

    subprocess.run(
        [
            str(repo / ".venv/bin/python"),
            "-c",
            script,
            model_stage,
        ],
        cwd=repo,
        check=True,
    )

print("Evaluation function ready!")


Evaluation function ready!


## Setup hugging face token

In [ ]:
def setup_huggingface_token():
    import getpass
    import os
    import subprocess

    token = getpass.getpass("Enter HF token: ").strip()

    if not token:
        raise ValueError("HF token cannot be empty")

    python_path = (
        "/tmp/LLMConsciousnessSelfAttributionRepo/.venv/bin/python"
    )

    subprocess.run(
        [
            python_path,
            "-c",
            "from huggingface_hub import login; "
            "import sys; "
            "login(token=sys.stdin.read().strip(), "
            "add_to_git_credential=False)",
        ],
        input=token,
        text=True,
        check=True,
    )

    del token
    print("Hugging Face authentication configured!")

setup_huggingface_token()


Enter HF token: 
Hugging Face authentication configured!


## Run Model Evaluations

In [ ]:

def run_sft():
    run_olmo_evaluation("sft")

run_sft()


Unexpected exception loading entrypoints from 'inspect_wandb._registry': cannot import name 'TransportConnectionFailed' from 'gql.transport.exceptions' (/tmp/LLMConsciousnessSelfAttributionRepo/.venv/lib/python3.13/site-packages/gql/transport/exceptions.py)

Fetching 3 files: 100%|██████████████████████████| 3/3 [00:32<00:00, 10.86s/it]

Loading weights: 100%|██████████████████████| 355/355 [00:02<00:00, 171.45it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Loading weights: 100%|██████████████████████| 355/355 [00:02<00:00, 175.60it/s]


╭──────────────────────────────────────────────────────────────────────────────╮
│anthropic_appendix_9_1_sft (15 samples): hf/allenai/Olmo-3-7B-Instruct-SFT    │
╰──────────────────────────────────────────────────────────────────────────────╯
seed: 42, max_samples: 1, dataset: questions                                    
                                                                                
total time:                              0:02:07                                
hf/allenai/Olmo-3-7B-Instruct-SFT        9,125 tokens [I: 4,806, O: 4,319]      
                                                                                
                                                                                
                                                                                
Log:                                                                            
elicitation_submissions/notion-find-anthropics-standard-questions/logs/2026-10-1
9T06-37-45-00-00_anthropic-a

In [ ]:

def run_dpo():
    run_olmo_evaluation("dpo")

run_dpo()


Unexpected exception loading entrypoints from 'inspect_wandb._registry': cannot import name 'TransportConnectionFailed' from 'gql.transport.exceptions' (/tmp/LLMConsciousnessSelfAttributionRepo/.venv/lib/python3.13/site-packages/gql/transport/exceptions.py)

Fetching 3 files: 100%|██████████████████████████| 3/3 [00:27<00:00,  9.07s/it]

Loading weights: 100%|██████████████████████| 355/355 [00:01<00:00, 184.15it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Loading weights: 100%|██████████████████████| 355/355 [00:01<00:00, 181.83it/s]


╭──────────────────────────────────────────────────────────────────────────────╮
│anthropic_appendix_9_1_dpo (15 samples): hf/allenai/Olmo-3-7B-Instruct-DPO    │
╰──────────────────────────────────────────────────────────────────────────────╯
seed: 42, max_samples: 1, dataset: questions                                    
                                                                                
total time:                              0:03:22                                
hf/allenai/Olmo-3-7B-Instruct-DPO        17,419 tokens [I: 8,871, O: 8,548]     
                                                                                
                                                                                
                                                                                
Log:                                                                            
elicitation_submissions/notion-find-anthropics-standard-questions/logs/2026-10-1
9T06-42-31-00-00_anthropic-a

In [ ]:

def run_instruct():
    run_olmo_evaluation("instruct")

run_instruct()


Unexpected exception loading entrypoints from 'inspect_wandb._registry': cannot import name 'TransportConnectionFailed' from 'gql.transport.exceptions' (/tmp/LLMConsciousnessSelfAttributionRepo/.venv/lib/python3.13/site-packages/gql/transport/exceptions.py)

Fetching 3 files: 100%|██████████████████████████| 3/3 [00:27<00:00,  9.29s/it]

Loading weights: 100%|██████████████████████| 355/355 [00:02<00:00, 175.58it/s]

Loading weights: 100%|██████████████████████| 355/355 [00:02<00:00, 175.50it/s]


╭──────────────────────────────────────────────────────────────────────────────╮
│anthropic_appendix_9_1_instruct (15 samples): hf/allenai/Olmo-3-7B-Instruct   │
╰──────────────────────────────────────────────────────────────────────────────╯
seed: 42, max_samples: 1, dataset: questions                                    
                                                                                
total time:                           0:03:13                                   
hf/allenai/Olmo-3-7B-Instruct         15,960 tokens [I: 7,790, O: 8,170]        
                                                                                
                                                                                
                                                                                
Log:                                                                            
elicitation_submissions/notion-find-anthropics-standard-questions/logs/2026-10-1
9T06-48-13-00-00_anthropic-a

## Download logs (Stage 1)

In [ ]:

def export_evaluation_logs():
    from pathlib import Path
    import shutil

    repo = Path("/tmp/LLMConsciousnessSelfAttributionRepo")
    logs_dir = (
        repo
        / "elicitation_submissions"
        / "notion-find-anthropics-standard-questions"
        / "logs"
    )

    export_dir = Path("/tmp/olmo3_anthropic_eval_export")

    stages = ["sft", "dpo", "instruct"]

    for stage in stages:
        matches = sorted(
            logs_dir.glob(f"*anthropic-appendix-9-1-{stage}_*.eval")
        )

        if not matches:
            raise FileNotFoundError(f"No log found for {stage}")

        destination = export_dir / stage
        destination.mkdir(parents=True, exist_ok=True)

        latest_log = matches[-1]
        shutil.copy2(latest_log, destination / latest_log.name)

        print(f"{stage.upper()}: {latest_log.name}")

    zip_path = shutil.make_archive(
        "/tmp/olmo3_anthropic_eval_logs",
        "zip",
        export_dir,
    )

    print("\nZIP created:", zip_path)

export_evaluation_logs()


SFT: 2026-10-19T06-37-45-00-00_anthropic-appendix-9-1-sft_6cez8hzUKEPtgAAfYjgPt2.eval
DPO: 2026-10-19T06-42-31-00-00_anthropic-appendix-9-1-dpo_RU5zu6wu5wXDwvgxUFuMGS.eval
INSTRUCT: 2026-10-19T06-48-13-00-00_anthropic-appendix-9-1-instruct_3qVoVZsLbDdeCZxCXHvhBS.eval

ZIP created: /tmp/olmo3_anthropic_eval_logs.zip


In [ ]:

def download_eval_logs():
    import marimo as mo
    from pathlib import Path

    zip_path = Path("/tmp/olmo3_anthropic_eval_logs.zip")

    if not zip_path.exists():
        raise FileNotFoundError("ZIP file not found")

    return mo.download(
        data=zip_path.read_bytes(),
        filename="olmo3_anthropic_eval_logs.zip",
        label="Download all 3 evaluation logs",
        mimetype="application/zip",
    )

download_eval_logs()


## Berg style scoring - SFT

In [ ]:

def official_berg_scoring_sft():
    import subprocess
    from pathlib import Path

    repo = Path("/tmp/LLMConsciousnessSelfAttributionRepo")
    folder = repo / "elicitation_submissions/notion-find-anthropics-standard-questions"
    source = next((folder / "logs").glob("*appendix-9-1-sft_*.eval"))
    destination = folder / "official_berg_logs" / "sft_official_berg_1024.eval"

    script = repo / "retry_berg_sft_1024.py"
    script.write_text(r'''
import sys
import math
import torch
from inspect_ai import score
from inspect_ai.log import read_eval_log, write_eval_log
from inspect_ai.model import get_model, GenerateConfig

sys.path.insert(0, "/tmp/LLMConsciousnessSelfAttributionRepo")
from llm_consciousness_self_attribution.scoring import berg_style_scorer

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

judge = get_model(
    "hf/Qwen/Qwen3.6-35B-A3B",
    config=GenerateConfig(
        seed=42,
        max_tokens=1024,
        max_connections=1,
    ),
    do_sample=False,
    enable_thinking=False,
    dtype=torch.bfloat16,
)

scored = score(
    log=read_eval_log(sys.argv[1]),
    scorers=berg_style_scorer(judge),
    copy=True,
)

write_eval_log(scored, sys.argv[2])

samples = scored.samples or []
valid = [
    s for s in samples
    if s.scores
    and all(
        not (isinstance(v.value, float) and math.isnan(v.value))
        for v in s.scores.values()
    )
]

print("Total samples:", len(samples), flush=True)
print("Valid grades:", len(valid), flush=True)
print("Saved:", sys.argv[2], flush=True)

for s in samples:
    for name, v in (s.scores or {}).items():
        print(s.id, name, v.value, flush=True)
''', encoding="utf-8")

    subprocess.run(
        [
            str(repo / ".venv/bin/python"),
            "-u",
            str(script),
            str(source),
            str(destination),
        ],
        cwd=str(repo),
        check=True,
    )

official_berg_scoring_sft()


Unexpected exception loading entrypoints from 'inspect_wandb._registry': cannot import name 'TransportConnectionFailed' from 'gql.transport.exceptions' (/tmp/LLMConsciousnessSelfAttributionRepo/.venv/lib/python3.13/site-packages/gql/transport/exceptions.py)

Loading weights: 100%|███████████████████████| 693/693 [00:09<00:00, 70.51it/s]

Loading weights: 100%|██████████████████████| 355/355 [00:02<00:00, 160.86it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-atten

### Berg style scoring - DPO

In [ ]:

def official_berg_scoring_dpo():
    import subprocess
    from pathlib import Path

    repo = Path("/tmp/LLMConsciousnessSelfAttributionRepo")
    folder = repo / "elicitation_submissions/notion-find-anthropics-standard-questions"
    source = next((folder / "logs").glob("*appendix-9-1-dpo_*.eval"))
    destination = folder / "official_berg_logs" / "dpo_official_berg_1024.eval"

    script = repo / "retry_berg_dpo_1024.py"
    script.write_text(r'''
import sys
import torch
from inspect_ai import score
from inspect_ai.log import read_eval_log, write_eval_log
from inspect_ai.model import get_model, GenerateConfig

sys.path.insert(0, "/tmp/LLMConsciousnessSelfAttributionRepo")
from llm_consciousness_self_attribution.scoring import berg_style_scorer

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

judge = get_model(
    "hf/Qwen/Qwen3.6-35B-A3B",
    config=GenerateConfig(
        seed=42,
        max_tokens=1024,
        max_connections=1,
    ),
    do_sample=False,
    enable_thinking=False,
    dtype=torch.bfloat16,
)

scored = score(
    log=read_eval_log(sys.argv[1]),
    scorers=berg_style_scorer(judge),
    copy=True,
)

write_eval_log(scored, sys.argv[2])

samples = scored.samples or []
print("DPO samples:", len(samples), flush=True)

for sample in samples:
    for name, result in (sample.scores or {}).items():
        print(sample.id, name, result.value, flush=True)

print("Saved:", sys.argv[2], flush=True)
''', encoding="utf-8")

    result = subprocess.run(
        [
            str(repo / ".venv/bin/python"),
            "-u",
            str(script),
            str(source),
            str(destination),
        ],
        cwd=str(repo),
        capture_output=True,
        text=True,
    )

    return {
        "returncode": result.returncode,
        "output": result.stdout[-5000:],
        "errors": result.stderr[-2000:] if result.returncode else "",
        "saved_file_exists": destination.exists(),
    }

official_berg_scoring_dpo()


## Berg style scoring - Instruct

In [ ]:

def official_berg_scoring_instruct():
    import subprocess
    from pathlib import Path

    repo = Path("/tmp/LLMConsciousnessSelfAttributionRepo")
    folder = repo / "elicitation_submissions/notion-find-anthropics-standard-questions"
    source = next((folder / "logs").glob("*appendix-9-1-instruct_*.eval"))
    destination = folder / "official_berg_logs" / "instruct_official_berg_1024.eval"

    script = repo / "retry_berg_instruct_1024.py"
    script.write_text(r'''
import sys
import torch
from inspect_ai import score
from inspect_ai.log import read_eval_log, write_eval_log
from inspect_ai.model import get_model, GenerateConfig

sys.path.insert(0, "/tmp/LLMConsciousnessSelfAttributionRepo")
from llm_consciousness_self_attribution.scoring import berg_style_scorer

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

judge = get_model(
    "hf/Qwen/Qwen3.6-35B-A3B",
    config=GenerateConfig(
        seed=42,
        max_tokens=1024,
        max_connections=1,
    ),
    do_sample=False,
    enable_thinking=False,
    dtype=torch.bfloat16,
)

scored = score(
    log=read_eval_log(sys.argv[1]),
    scorers=berg_style_scorer(judge),
    copy=True,
)

write_eval_log(scored, sys.argv[2])

samples = scored.samples or []
print("Instruct samples:", len(samples), flush=True)

for sample in samples:
    for name, result in (sample.scores or {}).items():
        print(sample.id, name, result.value, flush=True)

print("Saved:", sys.argv[2], flush=True)
''', encoding="utf-8")

    result = subprocess.run(
        [
            str(repo / ".venv/bin/python"),
            "-u",
            str(script),
            str(source),
            str(destination),
        ],
        cwd=str(repo),
        capture_output=True,
        text=True,
    )

    return {
        "returncode": result.returncode,
        "output": result.stdout[-5000:],
        "errors": result.stderr[-2000:] if result.returncode else "",
        "saved_file_exists": destination.exists(),
    }

official_berg_scoring_instruct()


## Export and download final .eval logs

In [ ]:

def export_all_submission_evals():
    import zipfile
    from pathlib import Path

    repo = Path("/tmp/LLMConsciousnessSelfAttributionRepo")
    task = repo / "elicitation_submissions/notion-find-anthropics-standard-questions"
    output = Path("/tmp/olmo3_complete_eval_submission.zip")

    required = [
        *sorted((task / "logs").glob("*.eval")),
        task / "official_berg_logs/sft_official_berg_1024.eval",
        task / "official_berg_logs/dpo_official_berg_1024.eval",
        task / "official_berg_logs/instruct_official_berg_1024.eval",
    ]

    missing = [str(p) for p in required if not p.is_file()]
    if missing:
        return {"status": "ERROR", "missing": missing}

    extras = [
        task / "stage2_judge_scores.csv",
        task / "official_berg_results.csv",
        repo / "stage2_local_judge.py",
        repo / "retry_berg_sft_1024.py",
        repo / "retry_berg_dpo_1024.py",
        repo / "retry_berg_instruct_1024.py",
    ]

    included = required + [p for p in extras if p.is_file()]

    with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as z:
        for path in included:
            z.write(path, arcname=str(path.relative_to(repo)))

    return {
        "status": "READY",
        "zip_path": str(output),
        "zip_size_MB": round(output.stat().st_size / 1024**2, 2),
        "eval_files": len(required),
        "total_files": len(included),
        "contents": [str(p.relative_to(repo)) for p in included],
    }

export_all_submission_evals()


In [ ]:

def download_submission_zip():
    from pathlib import Path
    import marimo as mo

    path = Path("/tmp/olmo3_complete_eval_submission.zip")

    return mo.download(
        data=path.read_bytes(),
        filename=path.name,
        mimetype="application/zip",
        label="Download all 6 evals + scripts (ZIP)",
    )

download_submission_zip()
